# 5교시. 문서 자동화 웹 애플리케이션 기본 구현

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/05_streamlit_basic.ipynb)

**목표:** Streamlit 파일 입력·버튼·결과 화면을 Python 처리 함수와 연결합니다.

**결과물:** `app_05.py`

- 모든 필수 실습은 Google Colab에서 진행합니다.
- 학습자 API 키나 결제가 필요 없습니다.
- 실행이 막히면 교재에 포함된 완성 복구본으로 같은 실습을 계속합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
import importlib.metadata
import subprocess

required_streamlit = "1.60.0"
try:
    installed_streamlit = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    installed_streamlit = None

if installed_streamlit != required_streamlit:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            f"streamlit=={required_streamlit}",
        ]
    )

import streamlit
print("Streamlit:", streamlit.__version__)


In [ ]:
SAMPLE_OCR_TEXT = '샘플문구점\n거래일자: 2026-07-27\n연필 2개 × 1,000원 = 2,000원\n노트 1개 × 3,000원 = 3,000원\n합계: 5,000원\n'
SAMPLE_VLM_MARKDOWN = '# 샘플문구점\n\n거래일자: 2026-07-27\n\n| 품목 | 수량 | 단가 | 금액 |\n|---|---:|---:|---:|\n| 연필 | 2 | 1,000원 | 2,000원 |\n| 노트 | 1 | 3,000원 | 3,000원 |\n\n**합계: 5,000원**\n'
SAMPLE_RECEIPT = {'document_type': 'receipt', 'store_name': '샘플문구점', 'date': '2026-07-27', 'total_amount': 5000, 'items': [{'name': '연필', 'quantity': 2, 'unit_price': 1000, 'line_total': 2000}, {'name': '노트', 'quantity': 1, 'unit_price': 3000, 'line_total': 3000}], 'source_mode': 'mock'}


## 핵심 3개

1. Streamlit은 위에서 아래로 실행되는 Python 스크립트를 웹앱으로 보여 줍니다.
2. 파일 입력·실행 버튼·결과 영역을 처리 함수와 연결합니다.
3. Colab에서는 브라우저 서버를 열지 않고 AppTest로 화면 코드를 검증합니다.


In [ ]:
from textwrap import dedent

app_code = dedent(
    '''
    import streamlit as st

    SAMPLE_TEXT = "준비된 영수증 판독 결과"
    SAMPLE_JSON = {
        "store_name": "샘플문구점",
        "date": "2026-07-27",
        "items": [],
        "total_amount": 5000,
        "source_mode": "prepared",
    }

    st.title("영수증 Document AI 미니 앱")
    uploaded = st.file_uploader(
        "영수증 이미지 또는 PDF 한 장",
        type=["png", "jpg", "jpeg", "pdf"],
    )
    if st.button("준비 결과로 실행"):
        st.info("준비 결과를 사용했습니다.")
        st.text_area("판독 원문", SAMPLE_TEXT)
        st.json(SAMPLE_JSON)
    '''
).lstrip()
output_path = OUTPUT_DIR / "app_05.py"
output_path.write_text(app_code, encoding="utf-8")
print("저장 완료:", output_path)


## 실습. Colab에서 Streamlit 앱 검사


In [ ]:
from streamlit.testing.v1 import AppTest

app_test = AppTest.from_file(str(output_path)).run(timeout=20)
assert not app_test.exception
assert app_test.title[0].value == "영수증 Document AI 미니 앱"
assert len(app_test.file_uploader) == 1
assert len(app_test.button) == 1
print("Streamlit 화면 코드 검사 완료")


## 완성 복구본

빈칸 수정이 어려우면 완성된 `app_05.py`를 저장하고 AppTest 결과를 확인합니다.
실제 브라우저 서버나 공개 터널을 열지 않아도 이번 교시를 완료할 수 있습니다.
